# HEST coverage95 formal benchmark harness

?? notebook ???? formal benchmark ???????????? `E:/Morpho-FM/benchmark` ??????????????????? baseline ??????????????? HistoOmniST workspace ?? high-confidence HEST human Visium manifest?slide-level split ? coverage95 canonical gene symbols?

## Benchmark contract

??????? `scripts/hest_evaluate_benchmark_predictions.py`??? package entrypoint `histoomnist-eval-benchmark-predictions`?

Baseline ????? sharded predictions?

- ??? slide ???`{sample_id}_count.npy`?`{sample_id}_rate.npy`?`{sample_id}_log1p_count.npy` ? `{sample_id}_log1p_rate.npy`?
- ?? `.npy` ??? `n_spots x n_genes`?
- spot ??????? manifest ? processed slide arrays ? valid spot ?????
- ? gene ???? coverage95 target ??????? `genes.txt`?????? gene symbol ?????? `n_common_genes`?
- truth?size factor?missing-gene mask?per-slide/per-organ labels ???? config ? manifest ?????? baseline notebook ??? truth?

In [1]:
from pathlib import Path
import json

import pandas as pd
from IPython.display import Markdown, display

CWD = Path.cwd()
ROOT = CWD if (CWD / "configs").exists() else CWD.parent
SMOKE = ROOT / "results" / "hest1k_human_visium_expression" / "benchmark_harness"

required = [
    SMOKE / "oracle_smoke_count" / "run_summary.json",
    SMOKE / "oracle_shard_count_eval" / "run_summary.json",
]
missing = [str(path.relative_to(ROOT)) for path in required if not path.exists()]
assert not missing, missing

oracle_summary = json.loads(required[0].read_text(encoding="utf-8"))
shard_summary = json.loads(required[1].read_text(encoding="utf-8"))

summary_table = pd.DataFrame([
    {
        "check": "oracle_internal_truth",
        "oracle_smoke_test": oracle_summary["oracle_smoke_test"],
        "n_slides": oracle_summary["n_slides"],
        "n_common_genes": oracle_summary["n_common_genes"],
        **oracle_summary["gene_metrics"],
    },
    {
        "check": "sharded_prediction_loader",
        "oracle_smoke_test": shard_summary["oracle_smoke_test"],
        "n_slides": shard_summary["n_slides"],
        "n_common_genes": shard_summary["n_common_genes"],
        **shard_summary["gene_metrics"],
    },
])
summary_table

,check,oracle_smoke_test,n_slides,n_common_genes,mean_gene_pearson,median_gene_pearson,valid_genes
0,oracle_internal_truth,True,2,16942,1.0,1.0,16937
1,sharded_prediction_loader,False,1,128,1.0,1.0,128


## What the smoke tests prove

`oracle_internal_truth` ??? coverage95 target ???? prediction??? target loader?canonical symbol mapping?missing-gene mask ? metric ???`sharded_prediction_loader` ???????? `.npy + genes.txt` prediction bundle??????? baseline ??????? gene-order alignment ??????

??? harness smoke tests????? benchmark??? baseline ??????????/???? method predictions?

In [2]:
oracle_gene = pd.read_csv(SMOKE / "oracle_smoke_count" / "per_gene_metrics.csv")
shard_gene = pd.read_csv(SMOKE / "oracle_shard_count_eval" / "per_gene_metrics.csv")

display(Markdown("**Oracle internal truth: first 10 genes**"))
display(oracle_gene.head(10).round(4))

display(Markdown("**Sharded prediction loader: first 10 genes**"))
display(shard_gene.head(10).round(4))

**Oracle internal truth: first 10 genes**

,method,prediction_kind,gene,gene_index,n_obs,pearson,mae,rmse,true_mean,pred_mean,true_std,pred_std,detected_fraction
0,oracle_smoke_count,count,HSPB1,0,7893,1.0,0.0,0.0,2.5435,2.5435,4.2579,4.2579,0.7291
1,oracle_smoke_count,count,CTSD,1,7893,1.0,0.0,0.0,5.3601,5.3601,6.0625,6.0625,0.8022
2,oracle_smoke_count,count,APP,2,7893,1.0,0.0,0.0,6.2187,6.2187,7.1130,7.1130,0.8576
3,oracle_smoke_count,count,SQSTM1,3,7893,1.0,0.0,0.0,7.3218,7.3218,9.0101,9.0101,0.8731
4,oracle_smoke_count,count,RHOA,4,7893,1.0,0.0,0.0,3.6944,3.6944,4.8308,4.8308,0.7754
5,oracle_smoke_count,count,TPM1,5,7893,1.0,0.0,0.0,7.9198,7.9198,6.9422,6.9422,0.9237
6,oracle_smoke_count,count,MIF,6,7893,1.0,0.0,0.0,1.1219,1.1219,1.8647,1.8647,0.4724
7,oracle_smoke_count,count,PGAM1,7,7893,1.0,0.0,0.0,0.9965,0.9965,1.5030,1.5030,0.4741
8,oracle_smoke_count,count,APOE,8,7893,1.0,0.0,0.0,1.7352,1.7352,2.5582,2.5582,0.5675
9,oracle_smoke_count,count,CYCS,9,7893,1.0,0.0,0.0,1.8695,1.8695,2.5466,2.5466,0.6444


**Sharded prediction loader: first 10 genes**

,method,prediction_kind,gene,gene_index,n_obs,pearson,mae,rmse,true_mean,pred_mean,true_std,pred_std,detected_fraction
0,oracle_shard_count,count,HSPB1,0,3385,1.0,0.0,0.0,3.6744,3.6744,5.9676,5.9676,0.7516
1,oracle_shard_count,count,CTSD,1,3385,1.0,0.0,0.0,2.9705,2.9705,6.0597,6.0597,0.6310
2,oracle_shard_count,count,APP,2,3385,1.0,0.0,0.0,4.6904,4.6904,8.9717,8.9717,0.7468
3,oracle_shard_count,count,SQSTM1,3,3385,1.0,0.0,0.0,5.6694,5.6694,10.8813,10.8813,0.7699
4,oracle_shard_count,count,RHOA,4,3385,1.0,0.0,0.0,2.9064,2.9064,6.0533,6.0533,0.6337
5,oracle_shard_count,count,TPM1,5,3385,1.0,0.0,0.0,6.0671,6.0671,7.0666,7.0666,0.8883
6,oracle_shard_count,count,MIF,6,3385,1.0,0.0,0.0,0.6750,0.6750,1.8927,1.8927,0.2874
7,oracle_shard_count,count,PGAM1,7,3385,1.0,0.0,0.0,0.4854,0.4854,1.3053,1.3053,0.2505
8,oracle_shard_count,count,APOE,8,3385,1.0,0.0,0.0,0.8074,0.8074,1.8871,1.8871,0.3421
9,oracle_shard_count,count,CYCS,9,3385,1.0,0.0,0.0,1.2936,1.2936,2.9031,2.9031,0.4514


## Baseline adaptation notes from `E:/Morpho-FM/benchmark`

? benchmark notebooks ???????????? `pred_bag.npy` / `true_bag.npy`??? notebook ???? MAE?RMSE ? mean gene-wise Pearson??????? total-count normalization + log1p???? single-slice INT28 ? kidney transfer???????? debug????????????? high-confidence HEST coverage95 benchmark?

???????

1. ???? baseline ??? train/val/test slide split ???????? checkpoint?
2. ? test slides ??? valid spot ???? sharded `.npy` predictions?
3. ?? prediction kind?`count`?`rate`?`log1p_count` ? `log1p_rate`?
4. ???????? `per_gene_metrics.csv`?`overall_metrics.csv`?`per_slide_metrics.csv` ? `per_organ_metrics.csv`?
5. ?? headline ????? gene-wise Pearson?flattened per-organ/per-slide ??????????????

In [3]:
display(Markdown("**Example command for a baseline that exports count predictions**"))
print(
    "python scripts/hest_evaluate_benchmark_predictions.py "
    "--method-name histogene_coverage95 "
    "--prediction-kind count "
    "--prediction-root results/hest1k_human_visium_expression/benchmark_predictions/histogene_coverage95 "
    "--prediction-genes-path genes.txt "
    "--prediction-pattern {sample_id}_count.npy "
    "--splits test "
    "--out-dir results/hest1k_human_visium_expression/benchmark_results/histogene_coverage95"
)

**Example command for a baseline that exports count predictions**

python scripts/hest_evaluate_benchmark_predictions.py --method-name histogene_coverage95 --prediction-kind count --prediction-root results/hest1k_human_visium_expression/benchmark_predictions/histogene_coverage95 --prediction-genes-path genes.txt --prediction-pattern {sample_id}_count.npy --splits test --out-dir results/hest1k_human_visium_expression/benchmark_results/histogene_coverage95


## Next benchmark work

The harness is now ready for actual baseline predictions. The next engineering step is to adapt one method at a time, starting with the least invasive historical notebook path: HisToGene or HiST. Each method should write only local ignored predictions/results; the committed artifact should be the adapter code and an executed notebook/report that points to the generated metrics without uploading large arrays.